# Phase 3 — Step 9 (rebuild): 7-class subtype classification

**Why this notebook is the rebuild.** The v1 notebook averaged A1 / C1 / D1 features across hashes (forbidden by `WORKFLOW.md §3.3`). This rebuild uses the long-row + per-neuron probability aggregation protocol (`WORKFLOW.md §3.6`), the same data-layer correction applied in notebooks 02 / 03 / 04 v2.

**Scope (narrowed per discussion):** pure multiclass subtype classification, no layer-recovery diagnostic. The layer-recovery comparison from `PHASE3_PLANNING §4` is deferred until the metamodel/depth mismatch is clarified with the professor.

**Population.** The Phase 1 working population (8,895 cells, 13 scans, L1 dropped). Seven AIBS metamodel subtype classes:

`23P (n=4198), 4P (n=2845), 5P-IT (n=1169), 5P-ET (n=320), 5P-NP (n=26), 6P-IT (n=216), 6P-CT (n=121)`

**Severe imbalance — 162:1 between 23P and 5P-NP.** Every methodological choice below is justified against this imbalance, exactly as in the Phase 1 multi-class stage runs.

**Headline metrics (because no single number is enough at this imbalance):**
1. **7-class macro balanced accuracy** — primary, immune to imbalance.
2. **"Common-4 macro recall"** over `{23P, 4P, 5P-IT, 5P-ET}` (each ≥ 320 cells) — auxiliary, robust to rare-class instability.
3. **Per-class recall + support** — visible status of every rare class.
4. **7 × 7 confusion matrix** — biologically interpretable (e.g. is 5P-NP confused with 5P-IT, as transcriptomics would predict?).

**The same scan-only LOSO integrity check from notebook 03 v2** is run here too. If a one-hot of `session_key` decodes 7 classes above 1/7 ≈ 0.143 under LOSO, the protocol is broken and we stop.


## 1. Setup

In [1]:
from __future__ import annotations

import sys, time, json, warnings
from pathlib import Path

import numpy as np
import pandas as pd

REPO_ROOT = Path('..').resolve().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.exceptions import ConvergenceWarning
from sklearn.metrics import (balanced_accuracy_score, recall_score,
                             confusion_matrix, f1_score, classification_report)
warnings.filterwarnings('ignore', category=ConvergenceWarning)
warnings.filterwarnings('ignore', category=FutureWarning, module='sklearn')

from src.config import (PROCESSED_TABLES_DIR, PROCESSED_FEATURES_DIR,
                        PROCESSED_RESULTS_DIR, RANDOM_SEED, ensure_dirs)
from src.data.loaders import build_modeling_table
from src.eval.metrics import neuron_level_score, summarize_cv_runs
from src.features.tier_b import B_FEATURE_NAMES
from src.features.tier_c import C1_FEATURE_NAMES
from src.features.tier_d import D_FEATURE_NAMES

ensure_dirs()
np.random.seed(RANDOM_SEED)

DATA_TABLES   = REPO_ROOT / 'data' / 'processed' / 'tables'
DATA_FEATURES = REPO_ROOT / 'data' / 'processed' / 'features'
DATA_RESULTS  = REPO_ROOT / 'data' / 'processed' / 'results'

GKF_OUT     = DATA_RESULTS / 'phase3_multiclass_runs.parquet'
LOSO_OUT    = DATA_RESULTS / 'phase3_multiclass_loso.parquet'
PERSCAN_OUT = DATA_RESULTS / 'phase3_multiclass_loso_perscan.parquet'
WINNER_OUT  = DATA_RESULTS / 'phase3_multiclass_winner.json'

# Class order: depth-bin order then IT/ET/NP/CT
CLASSES = np.array(['23P', '4P', '5P-IT', '5P-ET', '5P-NP', '6P-IT', '6P-CT'])
COMMON_4 = np.array(['23P', '4P', '5P-IT', '5P-ET'])  # n>=320 each
N_FOLDS = 5

# Multiclass LOSO validity: at least N classes with >= MIN cells in held-out scan.
MULTICLASS_LOSO_MIN_CELLS_PER_CLASS = 5
MULTICLASS_LOSO_N_VALID_CLASSES = 4


## 2. Build long-row table on the full Phase-1 working population

In [2]:
# 2.1 A1 long via canonical loader (label = celltype_label = AIBS metamodel)
X_a1, y_a1, g_a1, f_a1, df_a1 = build_modeling_table(
    level='A1', blocks=['amp', 'shape'], label='celltype_label')

# 2.2 + B (inner join restricts to repeated hashes)
b_hash = pd.read_parquet(DATA_FEATURES / 'B_per_hash.parquet',
    columns=['nucleus_id','condition_hash'] + list(B_FEATURE_NAMES))
df_a1b = df_a1.merge(b_hash, on=['nucleus_id','condition_hash'],
                     how='inner', validate='one_to_one')

# 2.3 + C1
c1 = pd.read_parquet(DATA_FEATURES / 'C1_per_hash.parquet',
    columns=['nucleus_id','condition_hash'] + list(C1_FEATURE_NAMES))
df_a1bc1 = df_a1b.merge(c1, on=['nucleus_id','condition_hash'],
                        how='left', validate='one_to_one')

# 2.4 + D
d = pd.read_parquet(DATA_FEATURES / 'D_per_hash.parquet',
    columns=['condition_hash'] + list(D_FEATURE_NAMES))
df_full = df_a1bc1.merge(d, on='condition_hash', how='left', validate='many_to_one')

# 2.5 + G broadcast
G = pd.read_parquet(DATA_FEATURES / 'G_per_neuron.parquet')
g_cols = [c for c in G.columns if c.startswith('g_')]
df = df_full.merge(G[['nucleus_id'] + g_cols], on='nucleus_id',
                   how='left', validate='many_to_one').reset_index(drop=True)

print(f'long-row modelling table: {df.shape}')
print(f'  unique neurons: {df["nucleus_id"].nunique():,}')
print(f'  unique scans  : {df["session_key"].nunique()}')
print()
print('class counts (neurons):')
nd = df.drop_duplicates('nucleus_id')
print(nd['celltype_label'].value_counts().reindex(CLASSES).to_string())
print()
counts = nd['celltype_label'].value_counts().reindex(CLASSES)
print(f'imbalance ratio (max/min): {counts.max()/counts.min():.0f}:1')


long-row modelling table: (1209720, 168)
  unique neurons: 8,895
  unique scans  : 13

class counts (neurons):
celltype_label
23P      4198
4P       2845
5P-IT    1169
5P-ET     320
5P-NP      26
6P-IT     216
6P-CT     121

imbalance ratio (max/min): 161:1


## 3. Refresh GKF folds stratified by subtype

In [3]:
# StratifiedGroupKFold on per-neuron table; broadcast to long rows.
neu = df[['nucleus_id','celltype_label','session_key']].drop_duplicates('nucleus_id').reset_index(drop=True)
sgkf = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_SEED)
neu['gkf_fold_phase3'] = -1
for k, (_, te) in enumerate(sgkf.split(np.zeros((len(neu),1)),
                                       neu['celltype_label'].to_numpy(),
                                       neu['nucleus_id'].to_numpy())):
    neu.loc[te,'gkf_fold_phase3'] = k
df = df.merge(neu[['nucleus_id','gkf_fold_phase3']], on='nucleus_id',
              how='left', validate='many_to_one')

# CRITICAL audit: per-fold class counts. With n=26 for 5P-NP, some folds may have
# 0 NP test cells. neuron_level_score handles that (only classes present in y_true
# contribute to per-class recall), but we surface it explicitly.
print('=== per-fold test-set class counts (neurons) ===')
audit = []
for k in range(N_FOLDS):
    te_neu = neu[neu['gkf_fold_phase3']==k]
    counts_te = te_neu['celltype_label'].value_counts().reindex(CLASSES, fill_value=0)
    audit.append({'fold': k, **{c: int(counts_te[c]) for c in CLASSES}})
audit_df = pd.DataFrame(audit).set_index('fold')
audit_df.loc['TOTAL'] = audit_df.sum()
print(audit_df.to_string())
print()
zero_cells = (audit_df.loc[range(N_FOLDS)] == 0).any(axis=0)
if zero_cells.any():
    print('CLASSES WITH ZERO TEST CELLS IN AT LEAST ONE FOLD:', zero_cells[zero_cells].index.tolist())
else:
    print('All classes present in every test fold. OK.')


=== per-fold test-set class counts (neurons) ===
        23P    4P  5P-IT  5P-ET  5P-NP  6P-IT  6P-CT
fold                                                
0       828   575    235     70      4     40     27
1       840   571    238     61      4     44     21
2       839   559    234     66      4     50     27
3       858   561    225     59      8     46     22
4       833   579    237     64      6     36     24
TOTAL  4198  2845   1169    320     26    216    121

All classes present in every test fold. OK.


## 4. Define feature blocks + row arrays

In [4]:
amp_cols   = [c for c in df.columns if c.startswith('amp_')]
shape_cols = [c for c in df.columns if c.startswith('shape_')]
a1_cols    = amp_cols + shape_cols
b_cols     = list(B_FEATURE_NAMES)
c1_cols    = list(C1_FEATURE_NAMES)
d_cols     = list(D_FEATURE_NAMES)

BLOCKS = {
    'G':            list(g_cols),
    'A1+B':         a1_cols + b_cols,
    'A1+B+C1':      a1_cols + b_cols + c1_cols,
    'A1+B+C1+D1':   a1_cols + b_cols + c1_cols + d_cols,
    'G+B+C1':       list(g_cols) + b_cols + c1_cols,
}
for n, cols in BLOCKS.items():
    print(f'  {n:<12s} {len(cols):3d} features')

# Row arrays
y_row    = df['celltype_label'].to_numpy()
groups_r = df['nucleus_id'].to_numpy()
sess_r   = df['session_key'].to_numpy()
folds_r  = df['gkf_fold_phase3'].to_numpy().astype(np.int8)
ccabs_r  = df['cc_abs'].to_numpy(np.float64)
print(f'\nlong rows: {len(df):,}')


  G            116 features
  A1+B          27 features
  A1+B+C1       31 features
  A1+B+C1+D1    41 features
  G+B+C1       127 features

long rows: 1,209,720


## 5. Multiclass helpers

In [5]:
def make_pipeline(model_name: str) -> Pipeline:
    if model_name == 'LogReg':
        return Pipeline([
            ('impute', SimpleImputer(strategy='median')),
            ('scale',  RobustScaler()),
            ('clf',    LogisticRegression(
                penalty='l2', C=1.0, solver='lbfgs', max_iter=400,
                class_weight='balanced', n_jobs=-1, random_state=RANDOM_SEED)),
        ])
    if model_name == 'HGB':
        return Pipeline([
            ('clf', HistGradientBoostingClassifier(
                max_iter=100, max_depth=8, learning_rate=0.05,
                l2_regularization=1.0,
                early_stopping=True, n_iter_no_change=10,
                random_state=RANDOM_SEED)),
        ])
    raise ValueError(model_name)


def _residualize(X_tr, X_te, c_tr, c_te):
    med = np.nanmedian(c_tr)
    c_tr_f = np.where(np.isnan(c_tr), med, c_tr)
    c_te_f = np.where(np.isnan(c_te), med, c_te)
    Xt = X_tr.copy(); Xe = X_te.copy()
    for j in range(X_tr.shape[1]):
        f_tr = Xt[:, j]; m = ~np.isnan(f_tr)
        if m.sum() < 5: continue
        c_fit = c_tr_f[m, 0]; f_fit = f_tr[m]
        cm_, fm_ = c_fit.mean(), f_fit.mean()
        denom = ((c_fit - cm_) ** 2).sum()
        if denom < 1e-12: continue
        b = ((c_fit - cm_) * (f_fit - fm_)).sum() / denom
        a = fm_ - b * cm_
        Xt[:, j] = f_tr - (a + b * c_tr_f[:, 0])
        Xe[:, j] = Xe[:, j] - (a + b * c_te_f[:, 0])
    return Xt, Xe


def _score_neuron(sc):
    # Augment neuron_level_score output with our extra metrics.
    out = {
        'balanced_accuracy': float(sc['balanced_accuracy']),
        'macro_f1':          float(sc['macro_f1']),
        'n_neurons':         int(sc['n_neurons']),
    }
    # common-4 macro recall: only classes present in y_true that are in COMMON_4
    y_true = sc['y_neuron_true']; y_pred = sc['y_neuron_pred']
    present_c4 = [c for c in COMMON_4 if (y_true == c).sum() > 0]
    if len(present_c4) >= 2:
        out['balanced_accuracy_common4'] = float(
            recall_score(y_true, y_pred, labels=present_c4, average='macro', zero_division=0))
    else:
        out['balanced_accuracy_common4'] = float('nan')
    for cls in CLASSES:
        out[f'recall_{cls}'] = sc['per_class_recall'].get(cls, float('nan'))
    out['cm'] = sc['confusion_matrix'].tolist()
    return out


def gkf_run_long(X, y, groups, folds, model_name,
                 residualize=False, ccabs=None, classes=CLASSES) -> list[dict]:
    rows = []
    for k in range(N_FOLDS):
        tr = folds != k; te = folds == k
        if tr.sum() == 0 or te.sum() == 0: continue
        if residualize:
            Xt, Xe = _residualize(X[tr], X[te], ccabs[tr].reshape(-1,1), ccabs[te].reshape(-1,1))
        else:
            Xt, Xe = X[tr], X[te]
        pipe = make_pipeline(model_name)
        sw = compute_sample_weight('balanced', y[tr])
        pipe.fit(Xt, y[tr], clf__sample_weight=sw)
        proba = pipe.predict_proba(Xe)
        sc = neuron_level_score(y[te], proba, groups[te], pipe.classes_)
        m = _score_neuron(sc); m['fold'] = k
        rows.append(m)
    return rows


def loso_run_long(X, y, groups, sessions, model_name,
                  residualize=False, ccabs=None, classes=CLASSES) -> list[dict]:
    rows = []
    for sk in sorted(np.unique(sessions).tolist()):
        te = sessions == sk; tr = ~te
        if tr.sum() == 0 or te.sum() == 0: continue
        if len(np.unique(y[tr])) < 2: continue
        if residualize:
            Xt, Xe = _residualize(X[tr], X[te], ccabs[tr].reshape(-1,1), ccabs[te].reshape(-1,1))
        else:
            Xt, Xe = X[tr], X[te]
        pipe = make_pipeline(model_name)
        sw = compute_sample_weight('balanced', y[tr])
        pipe.fit(Xt, y[tr], clf__sample_weight=sw)
        proba = pipe.predict_proba(Xe)
        sc = neuron_level_score(y[te], proba, groups[te], pipe.classes_)
        m = _score_neuron(sc); m['held_out_scan'] = sk
        # validity for this scan
        nd = pd.Series(sc['y_neuron_true']).value_counts().reindex(CLASSES, fill_value=0)
        present_at_thresh = int((nd >= MULTICLASS_LOSO_MIN_CELLS_PER_CLASS).sum())
        m['n_classes_present_at_threshold'] = present_at_thresh
        m['valid_for_loso'] = bool(present_at_thresh >= MULTICLASS_LOSO_N_VALID_CLASSES)
        for cls in CLASSES:
            m[f'n_test_{cls}'] = int(nd[cls])
        rows.append(m)
    return rows


def aggregate_gkf(rows: list[dict]) -> dict:
    # Mean / std across the per-fold dicts.
    out = {}
    for k in ('balanced_accuracy', 'balanced_accuracy_common4', 'macro_f1'):
        v = np.array([r[k] for r in rows if not np.isnan(r[k])], dtype=float)
        out[k] = float(v.mean()) if len(v) else float('nan')
        out[f'{k}_std'] = float(v.std()) if len(v) else float('nan')
    for cls in CLASSES:
        v = np.array([r[f'recall_{cls}'] for r in rows
                      if not np.isnan(r[f'recall_{cls}'])], dtype=float)
        out[f'recall_{cls}_mean'] = float(v.mean()) if len(v) else float('nan')
    out['n_folds'] = len(rows)
    return out


def aggregate_loso(rows: list[dict]) -> dict:
    df_ps = pd.DataFrame(rows)
    out = {}
    for tag, sub in (('all', df_ps), ('valid', df_ps[df_ps['valid_for_loso']])):
        for k in ('balanced_accuracy','balanced_accuracy_common4'):
            v = sub[k].dropna().to_numpy(float)
            out[f'{k}_{tag}_mean'] = float(v.mean()) if len(v) else float('nan')
            out[f'{k}_{tag}_std']  = float(v.std())  if len(v) else float('nan')
        out[f'n_scans_{tag}'] = int(len(sub))
        for cls in CLASSES:
            v = sub[f'recall_{cls}'].dropna().to_numpy(float)
            out[f'recall_{cls}_{tag}_mean'] = float(v.mean()) if len(v) else float('nan')
    return out


print('Helpers ready.')


Helpers ready.


## 6. GKF main grid (5 blocks × 2 models)

In [6]:
gkf_summary = []
gkf_perfold = []
t0 = time.time()
for block_name, cols in BLOCKS.items():
    X = df[cols].to_numpy(np.float64)
    for model in ('LogReg', 'HGB'):
        rows = gkf_run_long(X, y_row, groups_r, folds_r, model)
        agg  = aggregate_gkf(rows)
        for r in rows:
            gkf_perfold.append({'family':'main','block':block_name,'model':model,**r})
        gkf_summary.append({'family':'main','block':block_name,'model':model,'cv':'gkf',
                            'n_features': X.shape[1], **agg})
        print(f'[{block_name:<12s} | {model:<6s} | GKF ] '
              f'bal_acc7={agg["balanced_accuracy"]:.3f}±{agg["balanced_accuracy_std"]:.3f} '
              f'| common4={agg["balanced_accuracy_common4"]:.3f} '
              f'| f1m={agg["macro_f1"]:.3f}')
print(f'\nGKF main grid done in {time.time()-t0:.1f}s')


[G            | LogReg | GKF ] bal_acc7=0.376±0.009 | common4=0.417 | f1m=0.284
[G            | HGB    | GKF ] bal_acc7=0.389±0.027 | common4=0.517 | f1m=0.401
[A1+B         | LogReg | GKF ] bal_acc7=0.249±0.008 | common4=0.204 | f1m=0.144
[A1+B         | HGB    | GKF ] bal_acc7=0.329±0.024 | common4=0.323 | f1m=0.213
[A1+B+C1      | LogReg | GKF ] bal_acc7=0.262±0.009 | common4=0.214 | f1m=0.157
[A1+B+C1      | HGB    | GKF ] bal_acc7=0.383±0.026 | common4=0.407 | f1m=0.276
[A1+B+C1+D1   | LogReg | GKF ] bal_acc7=0.258±0.006 | common4=0.212 | f1m=0.154
[A1+B+C1+D1   | HGB    | GKF ] bal_acc7=0.424±0.022 | common4=0.467 | f1m=0.314
[G+B+C1       | LogReg | GKF ] bal_acc7=0.404±0.028 | common4=0.395 | f1m=0.275
[G+B+C1       | HGB    | GKF ] bal_acc7=0.396±0.022 | common4=0.520 | f1m=0.410

GKF main grid done in 3878.6s


## 7. GKF baselines + cc_abs-residualized winner

In [7]:
# 7.1 Majority class (predicts 23P)
neuron_df = df.drop_duplicates('nucleus_id').reset_index(drop=True)
y_neu = neuron_df['celltype_label'].to_numpy()
maj = pd.Series(y_neu).value_counts().idxmax()
y_pred_maj = np.full_like(y_neu, maj)
bal_maj = balanced_accuracy_score(y_neu, y_pred_maj)
present_c4 = [c for c in COMMON_4 if (y_neu == c).sum() > 0]
bal_maj_c4 = recall_score(y_neu, y_pred_maj, labels=present_c4, average='macro', zero_division=0)
gkf_summary.append({'family':'baseline','block':'-','model':'majority','cv':'gkf','n_features':0,
                    'balanced_accuracy': float(bal_maj),'balanced_accuracy_std': 0.0,
                    'balanced_accuracy_common4': float(bal_maj_c4),
                    'balanced_accuracy_common4_std': 0.0,
                    'macro_f1': float(f1_score(y_neu, y_pred_maj, average='macro', labels=CLASSES, zero_division=0)),
                    'macro_f1_std': 0.0,
                    **{f'recall_{cls}_mean': 1.0 if cls==maj else 0.0 for cls in CLASSES},
                    'n_folds': 1})
print(f'[baseline | majority    | GKF ] bal_acc7 = {bal_maj:.3f} (predicts {maj})')

# 7.2 Uniform-random
rng = np.random.default_rng(RANDOM_SEED)
y_pred_u = rng.choice(CLASSES, size=len(y_neu))
bal_u = balanced_accuracy_score(y_neu, y_pred_u)
bal_u_c4 = recall_score(y_neu, y_pred_u, labels=present_c4, average='macro', zero_division=0)
gkf_summary.append({'family':'baseline','block':'-','model':'uniform-random','cv':'gkf','n_features':0,
                    'balanced_accuracy': float(bal_u),'balanced_accuracy_std': 0.0,
                    'balanced_accuracy_common4': float(bal_u_c4),
                    'balanced_accuracy_common4_std': 0.0,
                    'macro_f1': float(f1_score(y_neu, y_pred_u, average='macro', labels=CLASSES, zero_division=0)),
                    'macro_f1_std': 0.0,
                    **{f'recall_{cls}_mean': float((y_pred_u[y_neu==cls]==cls).mean()) if (y_neu==cls).any() else 0.0 for cls in CLASSES},
                    'n_folds': 1})
print(f'[baseline | uniform-rand | GKF ] bal_acc7 = {bal_u:.3f}')

# 7.3 Scan-only (long-row one-hot)
scan_dummies = pd.get_dummies(df['session_key'], prefix='scan').to_numpy(np.float64)
for model in ('LogReg','HGB'):
    rows = gkf_run_long(scan_dummies, y_row, groups_r, folds_r, model)
    agg = aggregate_gkf(rows)
    gkf_summary.append({'family':'baseline','block':'scan-only','model':model,'cv':'gkf',
                        'n_features': scan_dummies.shape[1], **agg})
    print(f'[baseline | scan-only   | GKF | {model:<6s}] bal_acc7 = '
          f'{agg["balanced_accuracy"]:.3f} ± {agg["balanced_accuracy_std"]:.3f}'
          f' | common4 = {agg["balanced_accuracy_common4"]:.3f}')

# 7.4 cc_abs-only
ccabs_X = df[['cc_abs']].to_numpy(np.float64)
for model in ('LogReg','HGB'):
    rows = gkf_run_long(ccabs_X, y_row, groups_r, folds_r, model)
    agg = aggregate_gkf(rows)
    gkf_summary.append({'family':'baseline','block':'cc_abs-only','model':model,'cv':'gkf',
                        'n_features':1, **agg})
    print(f'[baseline | cc_abs-only | GKF | {model:<6s}] bal_acc7 = '
          f'{agg["balanced_accuracy"]:.3f} ± {agg["balanced_accuracy_std"]:.3f}')

# 7.5 Identify winner from GKF main, then residualize
gkf_main_df = pd.DataFrame([r for r in gkf_summary if r['family']=='main'])
winner = gkf_main_df.sort_values('balanced_accuracy', ascending=False).iloc[0]
WINNER_BLOCK = winner['block']; WINNER_MODEL = winner['model']
print(f'\nGKF winner: {WINNER_BLOCK} | {WINNER_MODEL}, bal_acc7 = {winner["balanced_accuracy"]:.3f}')

X_w = df[BLOCKS[WINNER_BLOCK]].to_numpy(np.float64)
rows = gkf_run_long(X_w, y_row, groups_r, folds_r, WINNER_MODEL,
                    residualize=True, ccabs=ccabs_r)
s_resid_gkf = aggregate_gkf(rows)
gkf_summary.append({'family':'baseline','block':f'{WINNER_BLOCK} (resid cc_abs)',
                    'model':WINNER_MODEL,'cv':'gkf',
                    'n_features':len(BLOCKS[WINNER_BLOCK]),**s_resid_gkf})
print(f'[{WINNER_BLOCK} | {WINNER_MODEL} | GKF | cc_abs-resid] bal_acc7 = '
      f'{s_resid_gkf["balanced_accuracy"]:.3f} ± {s_resid_gkf["balanced_accuracy_std"]:.3f}'
      f'  (Δ = {s_resid_gkf["balanced_accuracy"] - winner["balanced_accuracy"]:+.3f})')


[baseline | majority    | GKF ] bal_acc7 = 0.143 (predicts 23P)
[baseline | uniform-rand | GKF ] bal_acc7 = 0.135
[baseline | scan-only   | GKF | LogReg] bal_acc7 = 0.356 ± 0.053 | common4 = 0.349
[baseline | scan-only   | GKF | HGB   ] bal_acc7 = 0.356 ± 0.053 | common4 = 0.349
[baseline | cc_abs-only | GKF | LogReg] bal_acc7 = 0.206 ± 0.019
[baseline | cc_abs-only | GKF | HGB   ] bal_acc7 = 0.231 ± 0.020

GKF winner: A1+B+C1+D1 | HGB, bal_acc7 = 0.424
[A1+B+C1+D1 | HGB | GKF | cc_abs-resid] bal_acc7 = 0.360 ± 0.026  (Δ = -0.064)


## 8. LOSO grid + baselines + residualized winner

In [8]:
loso_summary = []
loso_perscan = []
t0 = time.time()
for block_name, cols in BLOCKS.items():
    X = df[cols].to_numpy(np.float64)
    for model in ('LogReg','HGB'):
        rows = loso_run_long(X, y_row, groups_r, sess_r, model)
        agg  = aggregate_loso(rows)
        for r in rows: loso_perscan.append({'family':'main','block':block_name,'model':model,**r})
        loso_summary.append({'family':'main','block':block_name,'model':model,'cv':'loso',
                             'n_features': X.shape[1], **agg})
        print(f'[{block_name:<12s} | {model:<6s} | LOSO] '
              f'valid({agg["n_scans_valid"]}) bal_acc7 = '
              f'{agg["balanced_accuracy_valid_mean"]:.3f} ± {agg["balanced_accuracy_valid_std"]:.3f}'
              f' | common4 = {agg["balanced_accuracy_common4_valid_mean"]:.3f}')

# LOSO baselines
# 8.1 majority
maj_rows = []
for sk in sorted(np.unique(sess_r).tolist()):
    te = sess_r == sk
    nd = df[te].drop_duplicates('nucleus_id')
    y_te = nd['celltype_label'].to_numpy()
    counts_te = pd.Series(y_te).value_counts().reindex(CLASSES, fill_value=0)
    if (counts_te > 0).sum() >= 2:
        pred = np.full_like(y_te, maj)
        bal = balanced_accuracy_score(y_te, pred)
        c4_pres = [c for c in COMMON_4 if (y_te == c).sum() > 0]
        bal_c4 = recall_score(y_te, pred, labels=c4_pres, average='macro', zero_division=0) if len(c4_pres) >= 2 else float('nan')
    else:
        bal = float('nan'); bal_c4 = float('nan')
    pres = int((counts_te >= MULTICLASS_LOSO_MIN_CELLS_PER_CLASS).sum())
    maj_rows.append({
        'held_out_scan': sk, 'n_neurons': int(len(nd)),
        'balanced_accuracy': bal, 'balanced_accuracy_common4': bal_c4,
        'macro_f1': float('nan'),
        **{f'recall_{cls}': (1.0 if cls==maj else 0.0) for cls in CLASSES},
        **{f'n_test_{cls}': int(counts_te[cls]) for cls in CLASSES},
        'n_classes_present_at_threshold': pres,
        'valid_for_loso': bool(pres >= MULTICLASS_LOSO_N_VALID_CLASSES),
        'cm': None,
    })
agg = aggregate_loso(maj_rows)
loso_summary.append({'family':'baseline','block':'-','model':'majority','cv':'loso','n_features':0,**agg})
for r in maj_rows: loso_perscan.append({'family':'baseline','block':'-','model':'majority',**r})
print(f'\n[baseline | majority    | LOSO] valid bal_acc7 = {agg["balanced_accuracy_valid_mean"]:.3f}')

# 8.2 scan-only (must collapse to chance under LOSO)
for model in ('LogReg','HGB'):
    rows = loso_run_long(scan_dummies, y_row, groups_r, sess_r, model)
    agg = aggregate_loso(rows)
    loso_summary.append({'family':'baseline','block':'scan-only','model':model,'cv':'loso',
                         'n_features': scan_dummies.shape[1], **agg})
    for r in rows: loso_perscan.append({'family':'baseline','block':'scan-only','model':model,**r})
    print(f'[baseline | scan-only   | LOSO | {model:<6s}] valid bal_acc7 = '
          f'{agg["balanced_accuracy_valid_mean"]:.3f} (must be ~1/7={1/7:.3f})')

# 8.3 cc_abs-only LOSO
for model in ('LogReg','HGB'):
    rows = loso_run_long(ccabs_X, y_row, groups_r, sess_r, model)
    agg = aggregate_loso(rows)
    loso_summary.append({'family':'baseline','block':'cc_abs-only','model':model,'cv':'loso',
                         'n_features':1, **agg})
    for r in rows: loso_perscan.append({'family':'baseline','block':'cc_abs-only','model':model,**r})
    print(f'[baseline | cc_abs-only | LOSO | {model:<6s}] valid bal_acc7 = '
          f'{agg["balanced_accuracy_valid_mean"]:.3f} ± {agg["balanced_accuracy_valid_std"]:.3f}')

# 8.4 cc_abs-residualized winner under LOSO
rows = loso_run_long(X_w, y_row, groups_r, sess_r, WINNER_MODEL,
                     residualize=True, ccabs=ccabs_r)
agg_resid_loso = aggregate_loso(rows)
loso_summary.append({'family':'baseline','block':f'{WINNER_BLOCK} (resid cc_abs)',
                     'model':WINNER_MODEL,'cv':'loso',
                     'n_features':len(BLOCKS[WINNER_BLOCK]), **agg_resid_loso})
for r in rows: loso_perscan.append({'family':'baseline',
                                    'block':f'{WINNER_BLOCK} (resid cc_abs)',
                                    'model':WINNER_MODEL,**r})
print(f'\n[PRIMARY] {WINNER_BLOCK} | {WINNER_MODEL} | LOSO | cc_abs-resid : '
      f'valid bal_acc7 = {agg_resid_loso["balanced_accuracy_valid_mean"]:.3f} ± '
      f'{agg_resid_loso["balanced_accuracy_valid_std"]:.3f}')
print(f'\nLOSO done in {time.time()-t0:.1f}s')


/opt/anaconda3/envs/neuroscience/lib/python3.10/site-packages/sklearn/metrics/_classification.py:2801: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/opt/anaconda3/envs/neuroscience/lib/python3.10/site-packages/sklearn/metrics/_classification.py:2801: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/opt/anaconda3/envs/neuroscience/lib/python3.10/site-packages/sklearn/metrics/_classification.py:2801: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/opt/anaconda3/envs/neuroscience/lib/python3.10/site-packages/sklearn/metrics/_classification.py:2801: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/opt/anaconda3/envs/neuroscience/lib/python3.10/site-packages/sklearn/metrics/_classification.py:2801: UserWarning: y_pred contains classes not in y_true
  

[G            | LogReg | LOSO] valid(10) bal_acc7 = 0.278 ± 0.038 | common4 = 0.301


/opt/anaconda3/envs/neuroscience/lib/python3.10/site-packages/sklearn/metrics/_classification.py:2801: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/opt/anaconda3/envs/neuroscience/lib/python3.10/site-packages/sklearn/metrics/_classification.py:2801: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/opt/anaconda3/envs/neuroscience/lib/python3.10/site-packages/sklearn/metrics/_classification.py:2801: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/opt/anaconda3/envs/neuroscience/lib/python3.10/site-packages/sklearn/metrics/_classification.py:2801: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/opt/anaconda3/envs/neuroscience/lib/python3.10/site-packages/sklearn/metrics/_classification.py:2801: UserWarning: y_pred contains classes not in y_true
  

[G            | HGB    | LOSO] valid(10) bal_acc7 = 0.274 ± 0.043 | common4 = 0.365


/opt/anaconda3/envs/neuroscience/lib/python3.10/site-packages/sklearn/metrics/_classification.py:2801: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/opt/anaconda3/envs/neuroscience/lib/python3.10/site-packages/sklearn/metrics/_classification.py:2801: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/opt/anaconda3/envs/neuroscience/lib/python3.10/site-packages/sklearn/metrics/_classification.py:2801: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/opt/anaconda3/envs/neuroscience/lib/python3.10/site-packages/sklearn/metrics/_classification.py:2801: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/opt/anaconda3/envs/neuroscience/lib/python3.10/site-packages/sklearn/metrics/_classification.py:2801: UserWarning: y_pred contains classes not in y_true
  

...

/opt/anaconda3/envs/neuroscience/lib/python3.10/site-packages/sklearn/metrics/_classification.py:2801: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/opt/anaconda3/envs/neuroscience/lib/python3.10/site-packages/sklearn/metrics/_classification.py:2801: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/opt/anaconda3/envs/neuroscience/lib/python3.10/site-packages/sklearn/metrics/_classification.py:2801: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/opt/anaconda3/envs/neuroscience/lib/python3.10/site-packages/sklearn/metrics/_classification.py:2801: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/opt/anaconda3/envs/neuroscience/lib/python3.10/site-packages/sklearn/metrics/_classification.py:2801: UserWarning: y_pred contains classes not in y_true
  

[baseline | cc_abs-only | LOSO | HGB   ] valid bal_acc7 = 0.195 ± 0.063


/opt/anaconda3/envs/neuroscience/lib/python3.10/site-packages/sklearn/metrics/_classification.py:2801: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/opt/anaconda3/envs/neuroscience/lib/python3.10/site-packages/sklearn/metrics/_classification.py:2801: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/opt/anaconda3/envs/neuroscience/lib/python3.10/site-packages/sklearn/metrics/_classification.py:2801: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/opt/anaconda3/envs/neuroscience/lib/python3.10/site-packages/sklearn/metrics/_classification.py:2801: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/opt/anaconda3/envs/neuroscience/lib/python3.10/site-packages/sklearn/metrics/_classification.py:2801: UserWarning: y_pred contains classes not in y_true
  


[PRIMARY] A1+B+C1+D1 | HGB | LOSO | cc_abs-resid : valid bal_acc7 = 0.337 ± 0.076

LOSO done in 16336.7s


/opt/anaconda3/envs/neuroscience/lib/python3.10/site-packages/sklearn/metrics/_classification.py:2801: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


## 9. Summary tables

In [9]:
gkf_df  = pd.DataFrame(gkf_summary)
loso_df = pd.DataFrame(loso_summary)
gkf_df['bal7_str']     = gkf_df.apply(lambda r: f"{r['balanced_accuracy']:.3f} ± {r.get('balanced_accuracy_std',0):.3f}", axis=1)
gkf_df['common4_str']  = gkf_df.apply(lambda r: f"{r['balanced_accuracy_common4']:.3f}", axis=1)
loso_df['valid7_str']   = loso_df.apply(lambda r: f"{r['balanced_accuracy_valid_mean']:.3f} ± {r['balanced_accuracy_valid_std']:.3f} (n={int(r['n_scans_valid'])})", axis=1)
loso_df['valid_c4_str'] = loso_df.apply(lambda r: f"{r['balanced_accuracy_common4_valid_mean']:.3f}", axis=1)

print('=== GKF MAIN GRID — 7-class (sorted by 7-class bal_acc) ===')
print(gkf_df[gkf_df['family']=='main']
      .sort_values('balanced_accuracy', ascending=False)
      [['block','model','n_features','bal7_str','common4_str','macro_f1']]
      .to_string(index=False))
print()
print('=== GKF BASELINES ===')
print(gkf_df[gkf_df['family']=='baseline']
      [['block','model','n_features','bal7_str','common4_str','macro_f1']]
      .to_string(index=False))
print()
print('=== LOSO MAIN GRID — 7-class (sorted by valid mean) ===')
print(loso_df[loso_df['family']=='main']
      .sort_values('balanced_accuracy_valid_mean', ascending=False)
      [['block','model','n_features','valid7_str','valid_c4_str']]
      .to_string(index=False))
print()
print('=== LOSO BASELINES ===')
print(loso_df[loso_df['family']=='baseline']
      [['block','model','n_features','valid7_str','valid_c4_str']]
      .to_string(index=False))


=== GKF MAIN GRID — 7-class (sorted by 7-class bal_acc) ===
     block  model  n_features      bal7_str common4_str  macro_f1
A1+B+C1+D1    HGB          41 0.424 ± 0.022       0.467  0.313752
    G+B+C1 LogReg         127 0.404 ± 0.028       0.395  0.275437
    G+B+C1    HGB         127 0.396 ± 0.022       0.520  0.409682
         G    HGB         116 0.389 ± 0.027       0.517  0.401028
   A1+B+C1    HGB          31 0.383 ± 0.026       0.407  0.275504
         G LogReg         116 0.376 ± 0.009       0.417  0.284335
      A1+B    HGB          27 0.329 ± 0.024       0.323  0.212787
   A1+B+C1 LogReg          31 0.262 ± 0.009       0.214  0.156601
A1+B+C1+D1 LogReg          41 0.258 ± 0.006       0.212  0.154033
      A1+B LogReg          27 0.249 ± 0.008       0.204  0.144266

=== GKF BASELINES ===
                    block          model  n_features      bal7_str common4_str  macro_f1
                        -       majority           0 0.143 ± 0.000       0.250  0.091608
             

## 10. Per-class recall & confusion matrix — GKF & LOSO winners

In [10]:
# Per-class recall, GKF winner
gw = gkf_df.loc[(gkf_df['family']=='main') &
                (gkf_df['block']==WINNER_BLOCK) &
                (gkf_df['model']==WINNER_MODEL)].iloc[0]
print(f'GKF WINNER: {WINNER_BLOCK} | {WINNER_MODEL}')
print(f'  bal_acc7        : {gw["balanced_accuracy"]:.3f} ± {gw["balanced_accuracy_std"]:.3f}')
print(f'  common-4 bal_acc: {gw["balanced_accuracy_common4"]:.3f}')
print('  per-class recall (GKF):')
for cls in CLASSES:
    n = int((y_neu == cls).sum())
    print(f'    {cls:<6s} n={n:>5d}  recall = {gw[f"recall_{cls}_mean"]:.3f}')
print()

# Per-class recall, LOSO valid (same block/model)
lw = loso_df.loc[(loso_df['family']=='main') &
                 (loso_df['block']==WINNER_BLOCK) &
                 (loso_df['model']==WINNER_MODEL)].iloc[0]
print(f'LOSO valid (same block/model): {WINNER_BLOCK} | {WINNER_MODEL}')
print(f'  bal_acc7        : {lw["balanced_accuracy_valid_mean"]:.3f} ± {lw["balanced_accuracy_valid_std"]:.3f}')
print(f'  common-4 bal_acc: {lw["balanced_accuracy_common4_valid_mean"]:.3f}')
print('  per-class recall (LOSO valid mean across scans where the class appears):')
for cls in CLASSES:
    n = int((y_neu == cls).sum())
    val = lw.get(f'recall_{cls}_valid_mean', float('nan'))
    print(f'    {cls:<6s} n={n:>5d}  recall = {val:.3f}')

# Confusion matrix from GKF winner per-fold rows
print()
cm_total = np.sum([np.array(r['cm'])
                   for r in gkf_perfold
                   if r['block']==WINNER_BLOCK and r['model']==WINNER_MODEL], axis=0)
cm_df = pd.DataFrame(cm_total, index=[f'{c}_true' for c in CLASSES],
                     columns=[f'{c}_pred' for c in CLASSES])
print(f'GKF WINNER — confusion matrix (totals across 5 folds):')
print(cm_df.to_string())
print()
cm_norm = cm_total / cm_total.sum(axis=1, keepdims=True).clip(min=1)
cm_norm_df = pd.DataFrame(cm_norm.round(3),
                          index=[f'{c}_true' for c in CLASSES],
                          columns=[f'{c}_pred' for c in CLASSES])
print('Row-normalised (recall per row, fraction of true class predicted as col):')
print(cm_norm_df.to_string())


GKF WINNER: A1+B+C1+D1 | HGB
  bal_acc7        : 0.424 ± 0.022
  common-4 bal_acc: 0.467
  per-class recall (GKF):
    23P    n= 4198  recall = 0.605
    4P     n= 2845  recall = 0.494
    5P-IT  n= 1169  recall = 0.276
    5P-ET  n=  320  recall = 0.494
    5P-NP  n=   26  recall = 0.050
    6P-IT  n=  216  recall = 0.370
    6P-CT  n=  121  recall = 0.677

LOSO valid (same block/model): A1+B+C1+D1 | HGB
  bal_acc7        : 0.284 ± 0.069
  common-4 bal_acc: 0.286
  per-class recall (LOSO valid mean across scans where the class appears):
    23P    n= 4198  recall = 0.447
    4P     n= 2845  recall = 0.356
    5P-IT  n= 1169  recall = 0.150
    5P-ET  n=  320  recall = 0.192
    5P-NP  n=   26  recall = 0.056
    6P-IT  n=  216  recall = 0.326
    6P-CT  n=  121  recall = 0.581

GKF WINNER — confusion matrix (totals across 5 folds):
            23P_pred  4P_pred  5P-IT_pred  5P-ET_pred  5P-NP_pred  6P-IT_pred  6P-CT_pred
23P_true        2539      405         216         355          31

## 11. Save

In [11]:
gkf_df.to_parquet(GKF_OUT, index=False)
print(f'wrote {GKF_OUT}  ({GKF_OUT.stat().st_size/1024:.1f} KB, {len(gkf_df)} rows)')
loso_df.to_parquet(LOSO_OUT, index=False)
print(f'wrote {LOSO_OUT}  ({LOSO_OUT.stat().st_size/1024:.1f} KB, {len(loso_df)} rows)')

ps_save = pd.DataFrame(loso_perscan).copy()
ps_save['cm'] = ps_save['cm'].apply(lambda v: json.dumps(v) if v is not None else None)
ps_save.to_parquet(PERSCAN_OUT, index=False)
print(f'wrote {PERSCAN_OUT}  ({PERSCAN_OUT.stat().st_size/1024:.1f} KB, {len(ps_save)} rows)')

winner_meta = {
    'phase':         'phase3_step9_v2_longrow',
    'task':          'multiclass_7way',
    'protocol':      'long-row + per-neuron probability aggregation (WORKFLOW §3.6)',
    'classes':       CLASSES.tolist(),
    'common4':       COMMON_4.tolist(),
    'winner_block':  WINNER_BLOCK,
    'winner_model':  WINNER_MODEL,
    'gkf_winner_balanced_accuracy7':       float(gw['balanced_accuracy']),
    'gkf_winner_balanced_accuracy7_std':   float(gw['balanced_accuracy_std']),
    'gkf_winner_balanced_accuracy_common4':float(gw['balanced_accuracy_common4']),
    'gkf_resid_balanced_accuracy7':        float(s_resid_gkf['balanced_accuracy']),
    'loso_valid_balanced_accuracy7':       float(lw['balanced_accuracy_valid_mean']),
    'loso_valid_balanced_accuracy7_std':   float(lw['balanced_accuracy_valid_std']),
    'loso_valid_balanced_accuracy_common4':float(lw['balanced_accuracy_common4_valid_mean']),
    'loso_resid_balanced_accuracy7':       float(agg_resid_loso['balanced_accuracy_valid_mean']),
    'loso_resid_balanced_accuracy7_std':   float(agg_resid_loso['balanced_accuracy_valid_std']),
    'n_valid_loso_scans':                  int(agg_resid_loso['n_scans_valid']),
}
with open(WINNER_OUT, 'w') as f:
    json.dump(winner_meta, f, indent=2)
print(f'wrote {WINNER_OUT}')


wrote /Users/katiarusso/Documents/VS CODE/Classes/Neuroscience/Final Project/data/processed/results/phase3_multiclass_runs.parquet  (14.9 KB, 17 rows)
wrote /Users/katiarusso/Documents/VS CODE/Classes/Neuroscience/Final Project/data/processed/results/phase3_multiclass_loso.parquet  (23.2 KB, 16 rows)
wrote /Users/katiarusso/Documents/VS CODE/Classes/Neuroscience/Final Project/data/processed/results/phase3_multiclass_loso_perscan.parquet  (37.7 KB, 208 rows)
wrote /Users/katiarusso/Documents/VS CODE/Classes/Neuroscience/Final Project/data/processed/results/phase3_multiclass_winner.json


## 12. Step-9 summary

In [12]:
print('=== PHASE 3 STEP 9 (REBUILD) — 7-class subtype classification ===')
print(f'protocol         : long-row + per-neuron probability aggregation (WORKFLOW §3.6)')
print(f'population       : n={len(neuron_df)} cells, {df["session_key"].nunique()} scans')
print(f'class imbalance  : {counts.max()/counts.min():.0f}:1 (rarest = {counts.idxmin()} n={counts.min()})')
print()
print(f'GKF winner       : {WINNER_BLOCK} | {WINNER_MODEL}')
print(f'  bal_acc7       : {gw["balanced_accuracy"]:.3f} ± {gw["balanced_accuracy_std"]:.3f}')
print(f'  common-4 bal   : {gw["balanced_accuracy_common4"]:.3f}')
print(f'  cc_abs-resid   : {s_resid_gkf["balanced_accuracy"]:.3f}  '
      f'(Δ = {s_resid_gkf["balanced_accuracy"]-gw["balanced_accuracy"]:+.3f})')
print()
print(f'LOSO (same model): n_valid = {int(lw["n_scans_valid"])}')
print(f'  bal_acc7 valid : {lw["balanced_accuracy_valid_mean"]:.3f} ± {lw["balanced_accuracy_valid_std"]:.3f}')
print(f'  common-4 valid : {lw["balanced_accuracy_common4_valid_mean"]:.3f}')
print(f'  cc_abs-resid   : {agg_resid_loso["balanced_accuracy_valid_mean"]:.3f}  '
      f'(Δ = {agg_resid_loso["balanced_accuracy_valid_mean"]-lw["balanced_accuracy_valid_mean"]:+.3f})')
print()
print('LOSO baselines (valid mean):')
for r in loso_df[loso_df['family']=='baseline'].itertuples():
    label = f"{r.block} {r.model}"
    print(f'  {label:<32s} {r.balanced_accuracy_valid_mean:.3f}')
print()
print(f'Sanity (chance / 1-of-7) = {1/7:.3f}; LOSO scan-only must come back near this.')
print()
print(f'GKF -> LOSO drop on winner: {gw["balanced_accuracy"] - lw["balanced_accuracy_valid_mean"]:+.3f}')
print()
print('Layer-recovery diagnostic deferred until prof clarifies metamodel/depth mismatch (PHASE3_PLANNING §4).')


=== PHASE 3 STEP 9 (REBUILD) — 7-class subtype classification ===
protocol         : long-row + per-neuron probability aggregation (WORKFLOW §3.6)
population       : n=8895 cells, 13 scans
class imbalance  : 161:1 (rarest = 5P-NP n=26)

GKF winner       : A1+B+C1+D1 | HGB
  bal_acc7       : 0.424 ± 0.022
  common-4 bal   : 0.467
  cc_abs-resid   : 0.360  (Δ = -0.064)

LOSO (same model): n_valid = 10
  bal_acc7 valid : 0.284 ± 0.069
  common-4 valid : 0.286
  cc_abs-resid   : 0.337  (Δ = +0.053)

LOSO baselines (valid mean):
  - majority                       0.165
  scan-only LogReg                 0.165
  scan-only HGB                    0.145
  cc_abs-only LogReg               0.178
  cc_abs-only HGB                  0.195
  A1+B+C1+D1 (resid cc_abs) HGB    0.337

Sanity (chance / 1-of-7) = 0.143; LOSO scan-only must come back near this.

GKF -> LOSO drop on winner: +0.140

Layer-recovery diagnostic deferred until prof clarifies metamodel/depth mismatch (PHASE3_PLANNING §4).
